# TCGA-BRCA Source Supplement Review

This notebook is review-only. It loads the latest saved Clinical Supplement and Biospecimen Supplement acquisition outputs from disk and summarizes them for source audit.

It does not call the GDC API, download files, parse XML, build cohorts, choose endpoints, or perform modeling.

## Load the latest saved supplement run

This section confirms that the stable latest-manifest exists and points to a completed supplement acquisition run.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


def parse_json_array_cell(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    parsed = json.loads(text)
    if not isinstance(parsed, list):
        raise ValueError(f"Expected a JSON array cell, received: {text}")
    return [str(item).strip() for item in parsed if str(item).strip()]


def classify_filename_pattern(filename: str) -> str:
    if ".TCGA-" in filename:
        return filename.split(".TCGA-", 1)[0]
    return filename


repo_root = find_repo_root(Path.cwd())
latest_manifest_path = repo_root / "01-data" / "audit" / "tcga-brca" / "source" / "tcga_brca_source_supplements_latest.json"
if not latest_manifest_path.exists():
    raise FileNotFoundError(
        f"Latest supplement manifest not found: {latest_manifest_path}. Run the supplement fetch script first."
    )

latest_manifest = json.loads(latest_manifest_path.read_text(encoding="utf-8"))
metadata_tsv_path = repo_root / latest_manifest["metadata_tsv"]
run_log_path = repo_root / latest_manifest["run_log_json"]
results_root = repo_root / "09-trials" / "01-tcga-only-source-audited" / "05-results"
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_manifest]))

,updated_at_utc,run_id,gdc_data_release,gdc_tag,audit_run_directory,metadata_tsv,run_log_json,source_classes
0,2026-04-12T00:39:13Z,20260412T000556Z,"Data Release 45.0 - December 04, 2025",8.3.1,01-data/audit/tcga-brca/source/supplements/run...,01-data/audit/tcga-brca/source/supplements/run...,01-data/audit/tcga-brca/source/supplements/run...,{'clinical': {'manifest_path': '01-data/raw/tc...


## Load saved metadata and run log

This section reads the saved supplement metadata TSV and run log from disk. The notebook uses these saved outputs only.

In [2]:
metadata_df = pd.read_csv(metadata_tsv_path, sep="\t")
run_log = json.loads(run_log_path.read_text(encoding="utf-8"))

print(f"Metadata TSV: {metadata_tsv_path}")
print(f"Run log: {run_log_path}")
print(f"Rows loaded: {len(metadata_df):,}")
display(metadata_df.head())

Metadata TSV: d:\Projects\brcapath-rx\01-data\audit\tcga-brca\source\supplements\runs\20260412T000556Z\tcga_brca_source_supplements_metadata.tsv
Run log: d:\Projects\brcapath-rx\01-data\audit\tcga-brca\source\supplements\runs\20260412T000556Z\run_log.json
Rows loaded: 3,388


,source_class,file_id,file_name,md5sum,file_size,access,state,data_category,data_type,data_format,case_ids,case_submitter_ids,project_ids,case_count
0,clinical,00049989-fa21-48fb-8dda-710c0dd5932e,nationwidechildrens.org_clinical.TCGA-A2-A0CT.xml,2142a4afcc7ce48075329b799a161b84,75821,open,released,Clinical,Clinical Supplement,BCR XML,"[""378778d2-b331-4867-a93b-c64028c8b4c7""]","[""TCGA-A2-A0CT""]","[""TCGA-BRCA""]",1
1,clinical,004b6bd4-19d0-4b40-99ef-1a76313fe7a5,nationwidechildrens.org_clinical.TCGA-GM-A2DD.xml,733d513f9097ef9582808fce4194b938,69143,open,released,Clinical,Clinical Supplement,BCR XML,"[""b343bfe0-7c23-4c6a-8c84-9ee39db2ecda""]","[""TCGA-GM-A2DD""]","[""TCGA-BRCA""]",1
2,clinical,00a5e81c-cd67-483f-9d99-3c733b2ead38,nationwidechildrens.org_clinical.TCGA-D8-A1JM.xml,202ce16353db30a7db21a36577da253a,72392,open,released,Clinical,Clinical Supplement,BCR XML,"[""3e775c99-ceda-4246-8d6f-0f58ca5097c8""]","[""TCGA-D8-A1JM""]","[""TCGA-BRCA""]",1
3,clinical,00a6f39d-7656-47af-8168-a133c4cec430,nationwidechildrens.org_clinical.TCGA-BH-A18Q.xml,4f351efa68ddb501cef231354eb685f9,36601,open,released,Clinical,Clinical Supplement,BCR XML,"[""db4bc6aa-2e7d-4bcb-8519-a455f624d33b""]","[""TCGA-BH-A18Q""]","[""TCGA-BRCA""]",1
4,clinical,014f5ae1-5862-4165-9a3b-bba7bb08a527,nationwidechildrens.org_clinical.TCGA-C8-A12P.xml,11ab568978f524787ed32926b7c7cf01,50440,open,released,Clinical,Clinical Supplement,BCR XML,"[""abdc76db-f85e-4337-a57e-6d098789da03""]","[""TCGA-C8-A12P""]","[""TCGA-BRCA""]",1


## Validate source classes and counts

This section checks that both expected supplement source classes are present and saves a compact count table.

In [3]:
supplement_counts = (
    metadata_df.groupby(["source_class", "data_type"], dropna=False)
    .size()
    .reset_index(name="file_count")
    .sort_values(["source_class", "data_type"])
)
supplement_counts.to_csv(results_root / "08_source_supplement_counts.tsv", sep="\t", index=False)
display(supplement_counts)

,source_class,data_type,file_count
0,biospecimen,Biospecimen Supplement,2205
1,clinical,Clinical Supplement,1183


## Case coverage review

This section summarizes the number of linked cases and unique case submitters per source class.

In [4]:
coverage_rows = []
for source_class, subset in metadata_df.groupby("source_class"):
    unique_case_ids = set()
    unique_case_submitters = set()
    for case_ids in subset["case_ids"].apply(parse_json_array_cell):
        unique_case_ids.update(case_ids)
    for case_submitters in subset["case_submitter_ids"].apply(parse_json_array_cell):
        unique_case_submitters.update(case_submitters)
    coverage_rows.append(
        {
            "source_class": source_class,
            "file_count": int(len(subset)),
            "unique_case_id_count": int(len(unique_case_ids)),
            "unique_case_submitter_count": int(len(unique_case_submitters)),
        }
    )
case_coverage_df = pd.DataFrame(coverage_rows).sort_values("source_class")
case_coverage_df.to_csv(results_root / "09_source_supplement_case_coverage.tsv", sep="\t", index=False)
display(case_coverage_df)

,source_class,file_count,unique_case_id_count,unique_case_submitter_count
0,biospecimen,2205,1098,1098
1,clinical,1183,1098,1098


## Data format, access, and state review

These summaries highlight the source-level structure of the supplement layer without parsing file contents.

In [5]:
data_format_counts = (
    metadata_df.groupby(["source_class", "data_format"], dropna=False)
    .size()
    .reset_index(name="file_count")
    .sort_values(["source_class", "file_count", "data_format"], ascending=[True, False, True])
)
data_format_counts.to_csv(results_root / "10_source_supplement_data_format_counts.tsv", sep="\t", index=False)

access_state_counts = (
    metadata_df.groupby(["source_class", "access", "state"], dropna=False)
    .size()
    .reset_index(name="file_count")
    .sort_values(["source_class", "access", "state"])
)
access_state_counts.to_csv(results_root / "11_source_supplement_access_state_counts.tsv", sep="\t", index=False)

display(data_format_counts)
display(access_state_counts)

,source_class,data_format,file_count
2,biospecimen,BCR XML,1098
1,biospecimen,BCR SSF XML,1097
0,biospecimen,BCR Biotab,10
5,clinical,BCR XML,1097
4,clinical,BCR OMF XML,77
3,clinical,BCR Biotab,9


,source_class,access,state,file_count
0,biospecimen,open,released,2205
1,clinical,open,released,1183


## Case-count distribution and filename patterns

This section distinguishes per-case files from project-level source tables and saves deterministic filename-pattern summaries.

In [6]:
case_count_distribution = (
    metadata_df.assign(case_count_numeric=metadata_df["case_count"].astype(int))
    .groupby(["source_class", "case_count_numeric"], dropna=False)
    .size()
    .reset_index(name="file_count")
    .sort_values(["source_class", "case_count_numeric"])
)
case_count_distribution.to_csv(results_root / "12_source_supplement_case_count_distribution.tsv", sep="\t", index=False)

filename_patterns = (
    metadata_df.assign(filename_pattern=metadata_df["file_name"].astype(str).map(classify_filename_pattern))
    .groupby(["source_class", "filename_pattern"], dropna=False)
    .size()
    .reset_index(name="file_count")
    .sort_values(["source_class", "file_count", "filename_pattern"], ascending=[True, False, True])
)
filename_patterns.to_csv(results_root / "13_source_supplement_filename_patterns.tsv", sep="\t", index=False)

display(case_count_distribution)
display(filename_patterns)

,source_class,case_count_numeric,file_count
0,biospecimen,1,2195
1,biospecimen,1098,10
2,clinical,1,1174
3,clinical,1098,9


,source_class,filename_pattern,file_count
0,biospecimen,nationwidechildrens.org_biospecimen,1098
9,biospecimen,nationwidechildrens.org_ssf,1097
1,biospecimen,nationwidechildrens.org_biospecimen_aliquot_br...,1
2,biospecimen,nationwidechildrens.org_biospecimen_analyte_br...,1
3,biospecimen,nationwidechildrens.org_biospecimen_diagnostic...,1
4,biospecimen,nationwidechildrens.org_biospecimen_portion_br...,1
5,biospecimen,nationwidechildrens.org_biospecimen_protocol_b...,1
6,biospecimen,nationwidechildrens.org_biospecimen_sample_brc...,1
7,biospecimen,nationwidechildrens.org_biospecimen_shipment_p...,1
8,biospecimen,nationwidechildrens.org_biospecimen_slide_brca...,1


## Download status review

This section records whether raw transfer was requested and completed for each source class in the reviewed run.

In [7]:
download_status_rows = []
for source_class, payload in latest_manifest["source_classes"].items():
    download_status_rows.append(
        {
            "source_class": source_class,
            "download_requested": bool(payload["download_requested"]),
            "download_completed": bool(payload["download_completed"]),
            "manifest_path": payload["manifest_path"],
            "download_dir": payload["download_dir"],
            "download_log_path": payload["download_log_path"],
        }
    )
download_status_df = pd.DataFrame(download_status_rows).sort_values("source_class")
download_status_df.to_csv(results_root / "14_source_supplement_download_status.tsv", sep="\t", index=False)
display(download_status_df)

,source_class,download_requested,download_completed,manifest_path,download_dir,download_log_path
1,biospecimen,True,True,01-data/raw/tcga-brca/gdc/manifests/biospecime...,01-data/raw/tcga-brca/gdc/downloads/biospecime...,01-data/raw/tcga-brca/gdc/logs/biospecimen/202...
0,clinical,True,True,01-data/raw/tcga-brca/gdc/manifests/clinical/2...,01-data/raw/tcga-brca/gdc/downloads/clinical/2...,01-data/raw/tcga-brca/gdc/logs/clinical/202604...


## Review reminder

These outputs remain part of source acquisition and source audit only. They do not freeze a cohort, endpoint, or downstream analysis design.